In [ ]:
"""
Spell Correction Examples in Python
Author: Phd Abdelouaheb

This file demonstrates:
1) Simple edit-distance correction with pyspellchecker
2) TextBlob spelling correction
3) SymSpell fast dictionary-based correction
4) Contextual correction using transformers (BERT masked LM)
5) Synthetic typo generation utility
NOTE: Installation commands are commented out. Uncomment in your environment.
"""

# --- Setup (uncomment as needed) ---
# pip install pyspellchecker textblob symspellpy transformers torch

import re
import random

# ============== 1) pyspellchecker (basic edit-distance + frequency) ==============
try:
    from spellchecker import SpellChecker
    spell = SpellChecker()
    words = ["speling", "langauge", "prosessing"]
    print("\n[pyspellchecker] corrections:")
    for w in words:
        print(f"{w:12s} -> {spell.correction(w)} (candidates={spell.candidates(w)})")
except Exception as e:
    print("[pyspellchecker] not available:", e)

# ============== 2) TextBlob ==============
try:
    from textblob import Word
    print("\n[TextBlob] corrections:")
    for w in ["korrectud", "adres", "inteligent"]:
        print(f"{w:12s} -> {Word(w).correct()}")
except Exception as e:
    print("[TextBlob] not available:", e)

# ============== 3) SymSpell ==============
try:
    from symspellpy.symspellpy import SymSpell, Verbosity

    max_edit_distance_dictionary = 2
    prefix_length = 7
    sym_spell = SymSpell(max_edit_distance_dictionary, prefix_length)

    # Load frequency dictionary (English from https://github.com/wolfgarbe/SymSpell)
    # For demo, add a few words manually
    sym_spell.create_dictionary_entry("natural", 50)
    sym_spell.create_dictionary_entry("language", 50)
    sym_spell.create_dictionary_entry("processing", 50)

    input_term = "langauge"
    suggestions = sym_spell.lookup(input_term, Verbosity.CLOSEST, max_edit_distance=2)
    print("\n[SymSpell] suggestions for 'langauge':")
    for sug in suggestions:
        print(sug)
except Exception as e:
    print("[SymSpell] not available:", e)

# ============== 4) Transformers (BERT Masked LM for context) ==============
try:
    from transformers import pipeline
    fill_mask = pipeline("fill-mask", model="bert-base-uncased")
    sent = "I love natural langauge processing."
    # Replace suspected error with [MASK]
    masked = sent.replace("langauge", "[MASK]")
    print("\n[Transformers] masked LM suggestions:")
    for pred in fill_mask(masked)[:5]:
        print(pred)
except Exception as e:
    print("[Transformers] not available:", e)

# ============== 5) Synthetic typo generator ==============
def random_typo(word: str) -> str:
    if len(word) < 2:
        return word
    i = random.randint(0, len(word)-2)
    return word[:i] + word[i+1] + word[i] + word[i+2:]

sentence = "Spell correction is important in NLP"
tokens = sentence.split()
noisy = [random_typo(t) if random.random() < 0.3 else t for t in tokens]
print("\n[Synthetic] original:", sentence)
print("[Synthetic] noisy   :", " ".join(noisy))

print("\nDone.")
